In [1]:
import transformers

print("transformers version:", transformers.__version__)


/var/lib/datausers_jupyterhub/lfalconi_storage/miniforge3/envs/pyt-eqa-fge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers version: 5.15.1


In [2]:
import torch

print("torch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU actual:", torch.cuda.current_device())
    print("Nombre de la GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
    x = torch.randn(2, 3, device="cuda")
    print("Prueba de tensor en GPU:", x)
else:
    print("No hay CUDA disponible. PyTorch está usando CPU.")

"""Verifica el dispositivo antes de ejecutar la celda de Qwen.
Si la siguiente celda del modelo muestra 'cuda:0', entonces la GPU está activa."""


torch version: 2.13.0+cu130
CUDA disponible: True
Número de GPUs: 8
GPU actual: 0
Nombre de la GPU: NVIDIA A16
Prueba de tensor en GPU: tensor([[ 0.7569,  0.3625,  0.7620],
        [-1.4615,  1.1723,  0.6471]], device='cuda:0')


"Verifica el dispositivo antes de ejecutar la celda de Qwen.\nSi la siguiente celda del modelo muestra 'cuda:0', entonces la GPU está activa."

In [3]:
%%time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook requires a GPU.")
gpu_id = 0
device = torch.device(f"cuda:{gpu_id}")
torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-3B-Instruct"
print("Using GPU:", torch.cuda.get_device_name(gpu_id))

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": gpu_id},
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()

print("model device:", next(model.parameters()).device)
print("input device check will follow below")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)
print("input_ids device:", model_inputs["input_ids"].device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)


Using GPU: NVIDIA A16


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 277.99it/s]


model device: cuda:0
input device check will follow below
input_ids device: cuda:0
A large language model (LLM) is a type of artificial intelligence designed to understand and generate human-like text based on the patterns it learns from vast amounts of textual data. These models are essentially neural networks that have been trained on massive datasets, often containing billions or even trillions of words.

Key characteristics of large language models include:

1. **Scale**: They are typically trained on extremely large datasets, which allows them to learn complex patterns and relationships within natural language.
2. **Complexity**: The models themselves are quite complex, with many layers and parameters that enable them to handle a wide range of tasks, including but not limited to text generation, translation, summarization, question-answering, and more.
3. **Relevance**: They can be fine-tuned for specific domains or tasks by adjusting their training data and adding additional cont

In [4]:
%%time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook requires a GPU.")
gpu_id = 4
device = torch.device(f"cuda:{gpu_id}")
torch.cuda.empty_cache()

model_name = "google/gemma-3-1b-it"
print("Using GPU:", torch.cuda.get_device_name(gpu_id))

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": gpu_id},
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()

print("model device:", next(model.parameters()).device)
print("input device check will follow below")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)
print("input_ids device:", model_inputs["input_ids"].device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)


Using GPU: NVIDIA A16


Loading weights: 100%|██████████| 340/340 [00:00<00:00, 616.11it/s]


model device: cuda:4
input device check will follow below
input_ids device: cuda:4
Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence that’s been trained on massive amounts of text data – think the entire internet!** They’re essentially incredibly sophisticated computer programs that can understand and generate human-like text. 

**Here’s a breakdown of what makes them special:**

* **They learn patterns:** LLMs analyze patterns in language – how words are used together, what topics are related, and how sentences are structured.
* **They can generate text:**  You can give them a prompt (a question, a request, or even just a starting sentence), and they'll produce new text that seems coherent and relevant.
* **They can do a lot:**  They can write articles, translate languages, summarize text, answer questions, write different kinds of creative content, and even generate code!

**Think of them as really

In [5]:
def gpu_memory(device_id):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3
    total = torch.cuda.get_device_properties(device_id).total_memory / 1024**3
    print(
        f"cuda:{device_id} | "
        f"allocated: {allocated:.2f} GiB | "
        f"reserved: {reserved:.2f} GiB | "
        f"total: {total:.2f} GiB"
    )

gpu_memory(0)
gpu_memory(4)

cuda:0 | allocated: 0.01 GiB | reserved: 5.82 GiB | total: 14.61 GiB
cuda:4 | allocated: 1.87 GiB | reserved: 1.91 GiB | total: 14.61 GiB


# Releasing resources
Once finished we shall release the resource since PyTorch allocates the model in memory so it can be used from cache and inference be faster

In [6]:
import gc
import torch

# Remove Python references to GPU-resident objects
for name in (
    "model",
    "tokenizer",
    "model_inputs",
    "generated_ids",
    "response",
    "text",
    "messages",
):
    globals().pop(name, None)

# Force Python garbage collection, then release unused PyTorch cache
gc.collect()

for device_id in range(torch.cuda.device_count()):
    with torch.cuda.device(device_id):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()